# React-OT on Google Colab (GPU fallback)



[Open on GitHub](https://github.com/PabloAMC/Maillard/blob/qm-barriers/notebooks/react_ot_colab_gpu.ipynb) | [Open in Colab](https://colab.research.google.com/github/PabloAMC/Maillard/blob/qm-barriers/notebooks/react_ot_colab_gpu.ipynb)



Use this notebook **only** when the in-container CPU smoke gate (`scripts/run_react_ot_smoke.py`) fails for `torch_geometric` / CUDA reasons. This notebook provisions a clean GPU environment on Google Colab, installs React-OT (deepprinciple/react-ot), runs inference on a bundled set of reactant/product XYZ pairs, and packages the resulting TS guesses + summary JSON for download.



**Trust posture (do not relax):**



* React-OT is treated strictly as a *geometric* TS-seed generator.

* React-OT energies are never propagated as runtime barrier authority.

* Every promising seed must clear downstream Sella DFT + imaginary-mode validation in the main `maillard` pipeline before any barrier change is considered.



**Local preparation:** first run `python scripts/prepare_react_ot_colab_bundle.py` in the repo. That emits a single uploadable bundle containing `reactants.zip`, `products.zip`, and `manifest.json` for the CHON-eligible targets.



**Workflow:** open in Colab -> set the runtime to **GPU** (`Runtime` -> `Change runtime type` -> `GPU`) -> run cells top to bottom -> upload the bundle emitted by `scripts/prepare_react_ot_colab_bundle.py` -> download `react_ot_colab_artifacts.zip` -> import it locally with `python scripts/import_react_ot_colab_artifacts.py PATH/TO/react_ot_colab_artifacts.zip` or `./scripts/docker_maillard.sh react-ot-import-colab PATH/TO/react_ot_colab_artifacts.zip`.


In [ ]:
# Sanity check the runtime: GPU + CUDA must be available.
import subprocess
print(subprocess.check_output(['nvidia-smi'], text=True))

In [ ]:
# Pin the upstream commit / ref. Bump only after re-validating the smoke gate.
REACT_OT_REPO = 'https://github.com/deepprinciple/react-ot.git'
REACT_OT_REF = 'main'
TORCH_VERSION = '2.2.1'  # match Colab's CUDA wheels
PYG_WHEEL_INDEX = f'https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu121.html'  # adjust if Colab CUDA differs
CHECKPOINT_URL = 'https://zenodo.org/records/13131875/files/sb-pretrained.ckpt'
import os, pathlib
WORK = pathlib.Path('/content/react_ot_workspace')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print('workspace:', WORK)

In [ ]:
# Clone React-OT at the pinned ref.
import subprocess
if not (WORK / 'react-ot').exists():
    subprocess.check_call(['git', 'clone', REACT_OT_REPO, str(WORK / 'react-ot')])
subprocess.check_call(['git', '-C', str(WORK / 'react-ot'), 'checkout', REACT_OT_REF])

In [ ]:
# Install the GPU torch + torch_geometric stack expected by React-OT.
# Use subprocess instead of `%pip` so Python variables are honored reliably.

import importlib
import importlib.util
import subprocess
import sys

CUDA_TAG = 'cu121'


def pip_install(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *args])


pip_install(
    f'torch=={TORCH_VERSION}+{CUDA_TAG}',
    '--extra-index-url',
    f'https://download.pytorch.org/whl/{CUDA_TAG}',
)
pip_install(
    'torch_scatter',
    'torch_sparse',
    'torch_cluster',
    'torch_geometric',
    '-f',
    PYG_WHEEL_INDEX,
)
pip_install(
    'pytorch-lightning==2.4.0',
    'torchdiffeq',
    'pymatgen==2024.11.13',
    'ase==3.23.0',
    'numpy==1.26.4',
    'pandas==2.2.3',
    'networkx',
    'timm',
    'lmdb',
    'rich',
)

# Try the editable install first; if it fails (upstream uses oa_reactdiff.egg-info
# while the actual package dir is `reactot/`), fall back to sys.path injection.
try:
    pip_install('--no-deps', '-e', str(WORK / 'react-ot'))
except subprocess.CalledProcessError as exc:
    print(f'pip install -e failed ({exc}); falling back to sys.path injection.')


def ensure_reactot_importable():
    repo_root = str(WORK / 'react-ot')
    if importlib.util.find_spec('reactot') is None:
        if repo_root not in sys.path:
            sys.path.insert(0, repo_root)
        importlib.invalidate_caches()
    if importlib.util.find_spec('reactot') is None:
        raise ModuleNotFoundError(
            f'reactot still not importable. Check that {repo_root}/reactot/__init__.py exists '
            'and that earlier setup cells (clone) ran successfully.'
        )
    import reactot

    return reactot


reactot = ensure_reactot_importable()

import torch

print(
    {
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(),
        'reactot_module': reactot.__file__,
    }
)

In [ ]:
# Download the pretrained checkpoint.
import urllib.request
ckpt_path = WORK / 'sb-pretrained.ckpt'
if not ckpt_path.exists():
    urllib.request.urlretrieve(CHECKPOINT_URL, ckpt_path)
print('checkpoint:', ckpt_path, ckpt_path.stat().st_size, 'bytes')

In [ ]:
# Upload one bundle created locally by scripts/prepare_react_ot_colab_bundle.py.

from google.colab import files

import json

import shutil

import zipfile



uploads_dir = WORK / 'uploads'

bundle_dir = WORK / 'bundle'

uploads_dir.mkdir(exist_ok=True)

if bundle_dir.exists():

    shutil.rmtree(bundle_dir)

bundle_dir.mkdir(exist_ok=True)



uploaded = files.upload()

if len(uploaded) != 1:

    raise ValueError('Upload exactly one react_ot_colab_bundle.zip file.')



bundle_name, bundle_blob = next(iter(uploaded.items()))

bundle_path = uploads_dir / bundle_name

bundle_path.write_bytes(bundle_blob)



with zipfile.ZipFile(bundle_path, 'r') as zip_ref:

    zip_ref.extractall(bundle_dir)



manifest = json.loads((bundle_dir / 'manifest.json').read_text())

print(json.dumps({'bundle': bundle_name, 'targets': [row['target'] for row in manifest['targets']]}, indent=2))


In [ ]:
# Run React-OT on the uploaded bundle and repackage outputs using repo-friendly names.

import importlib
import importlib.util
import json
import shutil
import sys
import time
import traceback
from types import SimpleNamespace

if 'WORK' not in globals():
    raise RuntimeError('Notebook setup variables are missing; rerun the setup cells from the top.')
if 'bundle_dir' not in globals() or 'ckpt_path' not in globals():
    raise RuntimeError('Bundle or checkpoint variables are missing; rerun the upload/checkpoint cells.')
if not (WORK / 'react-ot').exists():
    raise FileNotFoundError(f'missing cloned repo: {WORK / "react-ot"}')

if importlib.util.find_spec('reactot') is None:
    repo_root = str(WORK / 'react-ot')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
if importlib.util.find_spec('reactot') is None:
    raise ModuleNotFoundError(
        'reactot still not importable; rerun the install cell at the top of the notebook.'
    )

from reactot.run_model import pred_ts

raw_output_dir = WORK / 'react_ot_raw_output'
results_dir = WORK / 'results'
if raw_output_dir.exists():
    shutil.rmtree(raw_output_dir)
if results_dir.exists():
    shutil.rmtree(results_dir)
raw_output_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

reactants_zip = bundle_dir / 'reactants.zip'
products_zip = bundle_dir / 'products.zip'
manifest = json.loads((bundle_dir / 'manifest.json').read_text())

opt = SimpleNamespace(
    batch_size=72,
    nfe=10,
    solver='ode',
    checkpoint_path=str(ckpt_path),
    order=1,
    diz='linear',
    method='midpoint',
    atol=1e-2,
    rtol=1e-2,
)

run_started = time.time()
run_status = 'completed'
run_error = None
try:
    pred_ts(str(reactants_zip), str(products_zip), opt, str(raw_output_dir))
except Exception as exc:
    run_status = 'inference_failed'
    run_error = {'error': repr(exc), 'traceback': traceback.format_exc()}

summaries = []
for entry in manifest['targets']:
    target = entry['target']
    raw_ts = raw_output_dir / f'{target}_ts.xyz'
    raw_rxn = raw_output_dir / f'{target}_rxn.xyz'
    seed_xyz = results_dir / f'{target}_react_ot_seed.xyz'
    summary_json = results_dir / f'{target}_react_ot_seed.json'
    if raw_ts.exists():
        shutil.copyfile(raw_ts, seed_xyz)
        payload = {
            'target': target,
            'status': 'ok',
            'checkpoint': str(ckpt_path),
            'solver': opt.solver,
            'nfe': opt.nfe,
            'source_bundle_entry': entry,
            'raw_ts_path': str(raw_ts),
            'raw_rxn_path': str(raw_rxn) if raw_rxn.exists() else None,
            'seed_xyz_path': str(seed_xyz),
        }
    else:
        payload = {
            'target': target,
            'status': run_status if run_status != 'completed' else 'missing_ts_output',
            'checkpoint': str(ckpt_path),
            'solver': opt.solver,
            'nfe': opt.nfe,
            'source_bundle_entry': entry,
        }
        if run_error is not None:
            payload.update(run_error)
    summary_json.write_text(json.dumps(payload, indent=2, sort_keys=True))
    summaries.append({'target': target, 'status': payload['status']})

pilot_manifest = {
    'status': run_status,
    'device': 'cuda' if __import__('torch').cuda.is_available() else 'cpu',
    'checkpoint': str(ckpt_path),
    'wall_seconds': time.time() - run_started,
    'targets': summaries,
}
if run_error is not None:
    pilot_manifest['error'] = run_error['error']
(results_dir / 'react_ot_pilot_manifest.json').write_text(json.dumps(pilot_manifest, indent=2, sort_keys=True))

print(json.dumps(pilot_manifest, indent=2))

In [ ]:
# Bundle artifacts and download. Unpack under results/computational_gap_refinement/ in the local repo.
import shutil
from google.colab import files
archive = shutil.make_archive(str(WORK / 'react_ot_colab_artifacts'), 'zip', root_dir=str(results_dir))
print('archive:', archive)
files.download(archive)